# Week 5: Building a Tiny Transformer Block from Scratch

Week 4 built the mechanism that decides **where to look** (multi-head attention). This
week adds the machinery that decides **what to do with what it found** — a Feed-Forward
Network, Layer Normalization, residual connections, and stacking multiple blocks — and
tests, honestly, whether any of it actually helps.

**Scope discipline for this week:** only the components listed in the assignment —
Feed-Forward Network, LayerNorm, residual + norm, stacking 2 blocks, a next-token head,
and comparing 3 models of increasing sophistication. No causal masking refinements, no
new attention variants, no optimizer changes beyond plain SGD — those stay parked for
later weeks.

## Part A- A Synthetic GPO Workflow Dataset

The assignment gives 5 base workflow sequences. Rather than just repeating them verbatim,
we treat them as evidence of **3 underlying workflow "families"**, and generate a few
hundred synthetic sequences by walking each family's pattern with small random variations
(how many times a cycle repeats, where a walk happens to end):

- **Family A (replenishment cycle):** `Order → Shipment → Receive → Restock → Inventory → Forecast → Order → ...` (repeats)
- **Family B (scenario path):** `Inventory → Forecast → Scenario → Contract → Purchase → Rebate → NCR`
- **Family C (PO path):** `PO → Shipment → Invoice → Reconcile`

Notice the deliberate overlap: `Forecast` appears in both A and B, and `Shipment` appears
in both A and C — with a **different correct next-word each time**, depending on which
family the sequence belongs to. This is the same kind of ambiguity Week 4 explored, just
now embedded in a richer, more realistic-feeling synthetic dataset.

In [ ]:

import numpy as np

states = ["Receive","Restock","Inventory","Forecast","Order","Shipment",
          "Scenario","Contract","PO","Invoice","Reconcile","Purchase","Rebate","NCR"]
state_to_id = {s: i for i, s in enumerate(states)}
id_to_state = {i: s for s, i in state_to_id.items()}
vocab_size = len(states)
print(f"Vocabulary size: {vocab_size}")

def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / np.sum(e, axis=axis, keepdims=True)

def gen_family_A(rng, min_loops=1, max_loops=2):
    cycle = ["Order","Shipment","Receive","Restock","Inventory","Forecast"]
    start = rng.choice(["Order","Receive"])
    n_loops = rng.integers(min_loops, max_loops+1)
    idx = cycle.index(start)
    total_steps = n_loops * len(cycle) + rng.integers(1, len(cycle))
    return [cycle[(idx+i) % len(cycle)] for i in range(total_steps)]

def gen_family_B(rng):
    return ["Inventory","Forecast","Scenario","Contract","Purchase","Rebate","NCR"]

def gen_family_C(rng):
    return ["PO","Shipment","Invoice","Reconcile"]

def generate_dataset(n_sequences=260, seed=11):
    rng = np.random.default_rng(seed)
    seqs = []
    for _ in range(n_sequences):
        family = rng.choice(["A","B","C"], p=[0.5, 0.25, 0.25])
        seqs.append({"A": gen_family_A, "B": gen_family_B, "C": gen_family_C}[family](rng))
    return seqs

sequences = generate_dataset(n_sequences=260)
print(f"\nGenerated {len(sequences)} synthetic sequences. Sample:")
for s in sequences[:5]:
    print(" ", s)
lengths = [len(s) for s in sequences]
print(f"\nLength range: {min(lengths)}-{max(lengths)}, mean {np.mean(lengths):.1f}")


Vocabulary size: 14

Generated 260 synthetic sequences. Sample:
  ['Receive', 'Restock', 'Inventory', 'Forecast', 'Order', 'Shipment', 'Receive', 'Restock', 'Inventory']
  ['Receive', 'Restock', 'Inventory', 'Forecast', 'Order', 'Shipment', 'Receive']
  ['PO', 'Shipment', 'Invoice', 'Reconcile']
  ['Receive', 'Restock', 'Inventory', 'Forecast', 'Order', 'Shipment', 'Receive', 'Restock', 'Inventory', 'Forecast']
  ['Inventory', 'Forecast', 'Scenario', 'Contract', 'Purchase', 'Rebate', 'NCR']

Length range: 4-17, mean 8.8


In [ ]:

def build_pairs(seqs):
    pairs = []
    for seq in seqs:
        for t in range(1, len(seq)):
            pairs.append((seq[:t], seq[t]))
    return pairs

train_pairs = build_pairs(sequences)
print(f"Total causal (prefix -> next word) training pairs: {len(train_pairs)}")


Total causal (prefix -> next word) training pairs: 2039


## Part B- A Fair Generalization Test (and a mistake we caught)

To genuinely test whether a model learned the *transition rule* rather than just
memorizing which sequences start where, we build a **held-out test set** using starting
words that never appear as the first token of any training sequence.

**Our first attempt used `Inventory` as an "unseen" start for Family A — this was a
mistake we caught before trusting the results.** `Inventory` is *also* Family B's own
genuine starting word, so a context like `[Inventory, Forecast]` is byte-for-byte
identical to a real Family B training example. No model, however deep, could resolve
that fairly — it's not a generalization test, it's an unwinnable trick question caused by
how we built the test set, not a real weakness in any architecture. We verified this
directly against the training data before writing anything further.

In [ ]:

targets_seen = set()
for ctx, tgt in train_pairs:
    if ctx == ["Inventory", "Forecast"]:
        targets_seen.add(tgt)
print("Targets seen in training whenever context is exactly ['Inventory','Forecast']:", targets_seen)
print("-> Only one target ever appears. Testing a different 'true' answer for this exact")
print("   context would not be a fair test of any model -- it's testing against data")
print("   that was never in the training distribution to begin with.")


Targets seen in training whenever context is exactly ['Inventory','Forecast']: {'Scenario'}
-> Only one target ever appears. Testing a different 'true' answer for this exact
   context would not be a fair test of any model -- it's testing against data
   that was never in the training distribution to begin with.


In [ ]:

# Fixed test set: "Restock" and "Shipment" as bare starting tokens never occur as the
# FIRST token of any training sequence (Family A only starts Order/Receive; Family C
# always starts PO, never bare Shipment) -- so these are genuine, fair generalization tests.
def gen_family_A_unseen_start(rng, start_word):
    cycle = ["Order","Shipment","Receive","Restock","Inventory","Forecast"]
    idx = cycle.index(start_word)
    n_loops = rng.integers(1, 2)
    total_steps = n_loops * len(cycle) + rng.integers(1, len(cycle))
    return [cycle[(idx+i) % len(cycle)] for i in range(total_steps)]

def generate_test_set(seed=999, n=40):
    rng = np.random.default_rng(seed)
    seqs = []
    for _ in range(n):
        fam = rng.choice(["A_unseen","B","C"], p=[0.5,0.25,0.25])
        if fam == "A_unseen":
            start = rng.choice(["Restock","Shipment"])
            seqs.append(gen_family_A_unseen_start(rng, start))
        elif fam == "B":
            seqs.append(gen_family_B(rng))
        else:
            seqs.append(gen_family_C(rng))
    return seqs

test_sequences = generate_test_set()
test_pairs = build_pairs(test_sequences)
print(f"Generalization test set: {len(test_sequences)} sequences, {len(test_pairs)} pairs")
print("Sample unseen-start sequences:")
for s in test_sequences[:4]:
    print(" ", s)


Generalization test set: 40 sequences, 245 pairs
Sample unseen-start sequences:
  ['PO', 'Shipment', 'Invoice', 'Reconcile']
  ['Restock', 'Inventory', 'Forecast', 'Order', 'Shipment', 'Receive', 'Restock', 'Inventory', 'Forecast', 'Order']
  ['PO', 'Shipment', 'Invoice', 'Reconcile']
  ['Restock', 'Inventory', 'Forecast', 'Order', 'Shipment', 'Receive', 'Restock', 'Inventory', 'Forecast', 'Order', 'Shipment']


## Part C- Building Blocks: FFN and LayerNorm

Week 4 already gives us multi-head attention. Two new pieces this week:

**Feed-Forward Network (FFN).** In plain words: *attention decides what information
matters; the FFN decides what to do with that information.* It's a small 2-layer network
applied independently to each position's vector — expand to a wider hidden size, apply
ReLU, then shrink back down:
```
x = Linear(d_model, 4*d_model)
x = ReLU(x)
x = Linear(4*d_model, d_model)
```
**Why expand before shrinking?** The wider hidden layer (4x here) gives the network more
room to combine and reshape the features attention produced before compressing them back
down — similar in spirit to why Week 1's hidden layer needed to be wider than a single
number to represent anything useful. Squeezing straight from `d_model` to `d_model` with
no expansion would leave much less room for the nonlinearity (ReLU) to do useful work.

**Layer Normalization.** After attention (and after the FFN), values can drift to
different scales. LayerNorm rescales each position's vector back to a consistent, stable
range (mean 0, unit variance, per row) before the next computation — the same instinct as
subtracting `max(logits)` before softmax back in Week 1, just applied more broadly.

In [ ]:

def mha_forward(X, hp, num_heads, head_dim):
    T, d = X.shape
    Qs, Ks, Vs, As, Hs = [], [], [], [], []
    for i in range(num_heads):
        Qi = X @ hp["WQ"][i]; Ki = X @ hp["WK"][i]; Vi = X @ hp["WV"][i]
        Si = (Qi @ Ki.T) / np.sqrt(head_dim)
        Ai = softmax(Si, axis=-1)
        Hi = Ai @ Vi
        Qs.append(Qi); Ks.append(Ki); Vs.append(Vi); As.append(Ai); Hs.append(Hi)
    Hcat = np.concatenate(Hs, axis=1)
    O = Hcat @ hp["WO"]
    return O, dict(X=X, Qs=Qs, Ks=Ks, Vs=Vs, As=As, Hs=Hs, Hcat=Hcat, T=T)

def mha_backward(dO, cache, hp, num_heads, head_dim):
    X, Qs, Ks, Vs, As, Hcat = cache["X"], cache["Qs"], cache["Ks"], cache["Vs"], cache["As"], cache["Hcat"]
    dWO = Hcat.T @ dO
    dHcat = dO @ hp["WO"].T
    dX_total = np.zeros_like(X)
    dWQ, dWK, dWV = [None]*num_heads, [None]*num_heads, [None]*num_heads
    for i in range(num_heads):
        dHi = dHcat[:, i*head_dim:(i+1)*head_dim]
        Ai, Vi, Qi, Ki = As[i], Vs[i], Qs[i], Ks[i]
        dAi = dHi @ Vi.T
        dVi = Ai.T @ dHi
        sum_dA_A = np.sum(dAi * Ai, axis=-1, keepdims=True)
        dSi = Ai * (dAi - sum_dA_A) / np.sqrt(head_dim)
        dQi = dSi @ Ki; dKi = dSi.T @ Qi
        dWQ[i] = X.T @ dQi; dWK[i] = X.T @ dKi; dWV[i] = X.T @ dVi
        dX_total += dQi @ hp["WQ"][i].T + dKi @ hp["WK"][i].T + dVi @ hp["WV"][i].T
    return dX_total, dict(WQ=dWQ, WK=dWK, WV=dWV, WO=dWO)

def layernorm_forward(X, gamma, beta, eps=1e-5):
    mu = X.mean(axis=1, keepdims=True)
    var = X.var(axis=1, keepdims=True)
    std_inv = 1.0/np.sqrt(var+eps)
    Xhat = (X-mu)*std_inv
    Y = gamma*Xhat + beta
    return Y, dict(Xhat=Xhat, std_inv=std_inv, gamma=gamma, d=X.shape[1])

def layernorm_backward(dY, cache):
    Xhat, std_inv, gamma, d = cache["Xhat"], cache["std_inv"], cache["gamma"], cache["d"]
    dgamma = np.sum(dY*Xhat, axis=0)
    dbeta = np.sum(dY, axis=0)
    dXhat = dY*gamma
    s1 = np.sum(dXhat, axis=1, keepdims=True)
    s2 = np.sum(dXhat*Xhat, axis=1, keepdims=True)
    dX = (std_inv/d) * (d*dXhat - s1 - Xhat*s2)
    return dX, dgamma, dbeta

def ffn_forward(X, W1, b1, W2, b2):
    Z1 = X @ W1 + b1
    A1 = np.maximum(Z1, 0)
    Z2 = A1 @ W2 + b2
    return Z2, dict(X=X, Z1=Z1, A1=A1)

def ffn_backward(dZ2, cache, W1, W2):
    X, Z1, A1 = cache["X"], cache["Z1"], cache["A1"]
    dW2 = A1.T @ dZ2
    db2 = np.sum(dZ2, axis=0)
    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * (Z1 > 0)
    dW1 = X.T @ dZ1
    db1 = np.sum(dZ1, axis=0)
    dX = dZ1 @ W1.T
    return dX, dW1, db1, dW2, db2

print("Building blocks defined: multi-head attention (Week 4), FFN, LayerNorm -- each with forward + backward.")


Building blocks defined: multi-head attention (Week 4), FFN, LayerNorm -- each with forward + backward.


## Part D- Assembling One Transformer Block

Exactly the diagram from the assignment:
```
Input -> Multi-Head Attention -> Add Residual -> LayerNorm -> Feed-Forward -> Add Residual -> LayerNorm
```
**Why the residual connections (`Add Residual`)?** Each sub-layer adds its output *on top
of* its input, rather than replacing it (`R1 = X + Attention(X)`, not `R1 = Attention(X)`).
This gives the network a "free" path for information to skip past a sub-layer that isn't
helping yet, especially early in training when the sub-layers' weights are still close to
random — it's much easier to learn a small useful *adjustment* to the input than to learn
the entire transformation from scratch.

In [ ]:

emb_dim = 8
num_heads = 2
head_dim = emb_dim // num_heads
ffn_hidden = 4*emb_dim
max_len = 20

def init_mha(rng):
    return dict(WQ=[rng.normal(0,0.15,(emb_dim,head_dim)) for _ in range(num_heads)],
                WK=[rng.normal(0,0.15,(emb_dim,head_dim)) for _ in range(num_heads)],
                WV=[rng.normal(0,0.15,(emb_dim,head_dim)) for _ in range(num_heads)],
                WO=rng.normal(0,0.15,(num_heads*head_dim, emb_dim)))

def init_ffn(rng):
    return dict(W1=rng.normal(0,0.15,(emb_dim, ffn_hidden)), b1=np.zeros(ffn_hidden),
                W2=rng.normal(0,0.15,(ffn_hidden, emb_dim)), b2=np.zeros(emb_dim))

def init_ln(rng):
    return dict(gamma=np.ones(emb_dim), beta=np.zeros(emb_dim))

def transformer_block_forward(X, block):
    A, mha_cache = mha_forward(X, block["mha"], num_heads, head_dim)
    R1 = X + A                                                    # residual
    N1, ln1_cache = layernorm_forward(R1, block["ln1"]["gamma"], block["ln1"]["beta"])
    F, ffn_cache = ffn_forward(N1, block["ffn"]["W1"], block["ffn"]["b1"], block["ffn"]["W2"], block["ffn"]["b2"])
    R2 = N1 + F                                                   # residual
    N2, ln2_cache = layernorm_forward(R2, block["ln2"]["gamma"], block["ln2"]["beta"])
    return N2, dict(mha_cache=mha_cache, ln1_cache=ln1_cache, ffn_cache=ffn_cache, ln2_cache=ln2_cache, N1=N1)

def transformer_block_backward(dN2, cache, block):
    dR2, dgamma2, dbeta2 = layernorm_backward(dN2, cache["ln2_cache"])
    dN1_a, dF = dR2.copy(), dR2.copy()                            # residual splits gradient both ways
    dN1_b, dW1, db1, dW2, db2 = ffn_backward(dF, cache["ffn_cache"], block["ffn"]["W1"], block["ffn"]["W2"])
    dN1 = dN1_a + dN1_b
    dR1, dgamma1, dbeta1 = layernorm_backward(dN1, cache["ln1_cache"])
    dX_a, dA = dR1.copy(), dR1.copy()                             # residual splits gradient both ways
    dX_b, mha_grads = mha_backward(dA, cache["mha_cache"], block["mha"], num_heads, head_dim)
    dX = dX_a + dX_b
    grads = dict(mha=mha_grads, ln1=dict(gamma=dgamma1, beta=dbeta1),
                 ffn=dict(W1=dW1, b1=db1, W2=dW2, b2=db2), ln2=dict(gamma=dgamma2, beta=dbeta2))
    return dX, grads

print("Transformer block (forward + backward) assembled: MHA -> +res -> LN -> FFN -> +res -> LN")


Transformer block (forward + backward) assembled: MHA -> +res -> LN -> FFN -> +res -> LN


### Sanity check before trusting any of this
Before training a from-scratch implementation this deep, we verify the backward pass
against numerical gradients — for the trickiest piece (LayerNorm's gamma) as well as
ordinary weight matrices, buried two blocks deep in the stack.

In [ ]:

def init_model(seed, kind):
    rng = np.random.default_rng(seed)
    p = dict(E=rng.normal(0,0.15,(vocab_size, emb_dim)),
             P=rng.normal(0,0.05,(max_len, emb_dim)),
             Wout=rng.normal(0,0.15,(emb_dim, vocab_size)), bout=np.zeros(vocab_size))
    if kind == "A":
        pass
    elif kind == "B":
        p["mha"] = init_mha(rng)
    elif kind == "C":
        p["block1"] = dict(mha=init_mha(rng), ln1=init_ln(rng), ffn=init_ffn(rng), ln2=init_ln(rng))
        p["block2"] = dict(mha=init_mha(rng), ln1=init_ln(rng), ffn=init_ffn(rng), ln2=init_ln(rng))
    return p

def embed_and_pos(ids, p):
    T = len(ids)
    return p["E"][ids] + p["P"][:T]

def forward_A(ids, p):
    X = embed_and_pos(ids, p)
    last = X[-1]
    probs = softmax(last @ p["Wout"] + p["bout"])
    return probs, dict(X=X, last=last)

def backward_A(ids, target_id, probs, cache, p):
    y = np.zeros(vocab_size); y[target_id]=1.0
    dlogits = probs - y
    dWout = np.outer(cache["last"], dlogits); dbout = dlogits
    dlast = p["Wout"] @ dlogits
    dX = np.zeros_like(cache["X"]); dX[-1] = dlast
    dE = np.zeros_like(p["E"]); dP = np.zeros_like(p["P"])
    for t, tid in enumerate(ids):
        dE[tid] += dX[t]; dP[t] += dX[t]
    return dict(E=dE, P=dP, Wout=dWout, bout=dbout)

def forward_B(ids, p):
    X = embed_and_pos(ids, p)
    O, mc = mha_forward(X, p["mha"], num_heads, head_dim)
    last = O[-1]
    probs = softmax(last @ p["Wout"] + p["bout"])
    return probs, dict(X=X, O=O, last=last, mc=mc)

def backward_B(ids, target_id, probs, cache, p):
    y = np.zeros(vocab_size); y[target_id]=1.0
    dlogits = probs - y
    dWout = np.outer(cache["last"], dlogits); dbout = dlogits
    dlast = p["Wout"] @ dlogits
    dO = np.zeros_like(cache["O"]); dO[-1] = dlast
    dX, mha_grads = mha_backward(dO, cache["mc"], p["mha"], num_heads, head_dim)
    dE = np.zeros_like(p["E"]); dP = np.zeros_like(p["P"])
    for t, tid in enumerate(ids):
        dE[tid] += dX[t]; dP[t] += dX[t]
    return dict(E=dE, P=dP, Wout=dWout, bout=dbout, mha=mha_grads)

def forward_C(ids, p):
    X = embed_and_pos(ids, p)
    N2_1, c1 = transformer_block_forward(X, p["block1"])
    N2_2, c2 = transformer_block_forward(N2_1, p["block2"])
    last = N2_2[-1]
    probs = softmax(last @ p["Wout"] + p["bout"])
    return probs, dict(X=X, N2_2=N2_2, c1=c1, c2=c2, last=last)

def backward_C(ids, target_id, probs, cache, p):
    y = np.zeros(vocab_size); y[target_id]=1.0
    dlogits = probs - y
    dWout = np.outer(cache["last"], dlogits); dbout = dlogits
    dlast = p["Wout"] @ dlogits
    dN2_2 = np.zeros_like(cache["N2_2"]); dN2_2[-1] = dlast
    dN2_1, grads2 = transformer_block_backward(dN2_2, cache["c2"], p["block2"])
    dX, grads1 = transformer_block_backward(dN2_1, cache["c1"], p["block1"])
    dE = np.zeros_like(p["E"]); dP = np.zeros_like(p["P"])
    for t, tid in enumerate(ids):
        dE[tid] += dX[t]; dP[t] += dX[t]
    return dict(E=dE, P=dP, Wout=dWout, bout=dbout, block1=grads1, block2=grads2)

print("Three models defined:")
print("  Model A: Embedding + Position -> Linear -> Prediction  (no attention at all)")
print("  Model B: Embedding + Position -> Multi-Head Attention -> Prediction  (Week 4, no FFN/LN/residual)")
print("  Model C: Embedding + Position -> 2 stacked Transformer blocks -> Prediction  (this week, full)")


Three models defined:
  Model A: Embedding + Position -> Linear -> Prediction  (no attention at all)
  Model B: Embedding + Position -> Multi-Head Attention -> Prediction  (Week 4, no FFN/LN/residual)
  Model C: Embedding + Position -> 2 stacked Transformer blocks -> Prediction  (this week, full)


In [ ]:

# Gradient check on Model C -- the deepest, riskiest implementation -- before trusting it
p = init_model(seed=42, kind="C")
ids = [state_to_id[w] for w in ["Order","Shipment","Receive","Restock","Inventory","Forecast"]]
target_id = state_to_id["Order"]
probs, cache = forward_C(ids, p)
grads = backward_C(ids, target_id, probs, cache, p)

def check(name, arr, grad_arr, i, j):
    eps = 1e-5
    orig = arr[i,j]
    arr[i,j] = orig + eps
    p_p, _ = forward_C(ids, p); l_p = -np.log(p_p[target_id]+1e-12)
    arr[i,j] = orig - eps
    p_m, _ = forward_C(ids, p); l_m = -np.log(p_m[target_id]+1e-12)
    arr[i,j] = orig
    numeric = (l_p - l_m) / (2*eps)
    print(f"  {name:28s} analytic={grad_arr[i,j]: .6f}  numeric={numeric: .6f}  diff={abs(numeric-grad_arr[i,j]):.2e}")

print("Gradient check, Model C (2-block Transformer):\n")
check("Wout[3,5]", p["Wout"], grads["Wout"], 3, 5)
check("block1.ffn.W1[2,10]", p["block1"]["ffn"]["W1"], grads["block1"]["ffn"]["W1"], 2, 10)
check("block1.mha.WO[1,4]", p["block1"]["mha"]["WO"], grads["block1"]["mha"]["WO"], 1, 4)
check("block2.ffn.W2[15,3]", p["block2"]["ffn"]["W2"], grads["block2"]["ffn"]["W2"], 15, 3)
check("E[Order,2]", p["E"], grads["E"], state_to_id["Order"], 2)

eps=1e-5
arr = p["block1"]["ln1"]["gamma"]; g = grads["block1"]["ln1"]["gamma"]
orig = arr[3]; arr[3]=orig+eps
p_p,_=forward_C(ids,p); l_p=-np.log(p_p[target_id]+1e-12)
arr[3]=orig-eps
p_m,_=forward_C(ids,p); l_m=-np.log(p_m[target_id]+1e-12)
arr[3]=orig
numeric=(l_p-l_m)/(2*eps)
label = "block1.ln1.gamma[3]"
print(f"  {label:28s} analytic={g[3]: .6f}  numeric={numeric: .6f}  diff={abs(numeric-g[3]):.2e}")
print("\nAll gradients match to within numerical precision -> backward pass is correct.")


Gradient check, Model C (2-block Transformer):

  Wout[3,5]                    analytic= 0.054319  numeric= 0.054319  diff=3.33e-12
  block1.ffn.W1[2,10]          analytic=-0.005248  numeric=-0.005248  diff=2.88e-11
  block1.mha.WO[1,4]           analytic= 0.002812  numeric= 0.002812  diff=1.22e-11
  block2.ffn.W2[15,3]          analytic= 0.000000  numeric= 0.000000  diff=0.00e+00
  E[Order,2]                   analytic=-0.007697  numeric=-0.007697  diff=1.28e-11
  block1.ln1.gamma[3]          analytic=-0.095734  numeric=-0.095734  diff=1.18e-12

All gradients match to within numerical precision -> backward pass is correct.


## Part E- Training All Three Models

To answer *"does architecture actually matter?"* we train three models of increasing
sophistication on the exact same data, same embedding size, same number of epochs:

- **Model A:** Embedding + Position → Linear → Prediction (no attention, no context mechanism at all beyond position)
- **Model B:** Embedding + Position → Multi-Head Attention → Prediction (Week 4's architecture, no FFN/LayerNorm/residual)
- **Model C:** Embedding + Position → 2 stacked Transformer blocks → Prediction (this week's full architecture)

In [ ]:

rng_sub = np.random.default_rng(3)
idx = rng_sub.choice(len(train_pairs), size=450, replace=False)
train_pairs_sub = [train_pairs[i] for i in idx]
idx2 = rng_sub.choice(len(test_pairs), size=min(150, len(test_pairs)), replace=False)
test_pairs_sub = [test_pairs[i] for i in idx2]
print(f"Training on {len(train_pairs_sub)} pairs (subsampled for speed), testing on {len(test_pairs_sub)} unseen pairs.")


Training on 450 pairs (subsampled for speed), testing on 150 unseen pairs.


### An early finding: Model B needed a smaller learning rate

Before settling on final hyperparameters, we tried training Model B (attention with **no**
LayerNorm) at the same learning rate as the others. **It diverged to `NaN` partway through
training.** Model C, which has LayerNorm after every sub-layer, trained stably at that
same learning rate without any issue.

This wasn't a bug we had to fix in the *math* — it's a direct, organic demonstration of
exactly what LayerNorm is supposed to buy us: without it, nothing keeps intermediate
values from growing unboundedly across training steps, and the model becomes numerically
fragile. We lowered Model B's learning rate (0.04, vs. 0.08 for A and C) purely to keep it
trainable at all — itself a finding worth reporting, not just a hyperparameter footnote.

In [ ]:

def apply_grads(p, g, lr, kind):
    p["E"] -= lr*g["E"]; p["P"] -= lr*g["P"]
    p["Wout"] -= lr*g["Wout"]; p["bout"] -= lr*g["bout"]
    if kind == "B":
        for i in range(num_heads):
            p["mha"]["WQ"][i] -= lr*g["mha"]["WQ"][i]
            p["mha"]["WK"][i] -= lr*g["mha"]["WK"][i]
            p["mha"]["WV"][i] -= lr*g["mha"]["WV"][i]
        p["mha"]["WO"] -= lr*g["mha"]["WO"]
    if kind == "C":
        for bname in ["block1","block2"]:
            gb, pb = g[bname], p[bname]
            for i in range(num_heads):
                pb["mha"]["WQ"][i] -= lr*gb["mha"]["WQ"][i]
                pb["mha"]["WK"][i] -= lr*gb["mha"]["WK"][i]
                pb["mha"]["WV"][i] -= lr*gb["mha"]["WV"][i]
            pb["mha"]["WO"] -= lr*gb["mha"]["WO"]
            pb["ln1"]["gamma"] -= lr*gb["ln1"]["gamma"]; pb["ln1"]["beta"] -= lr*gb["ln1"]["beta"]
            pb["ln2"]["gamma"] -= lr*gb["ln2"]["gamma"]; pb["ln2"]["beta"] -= lr*gb["ln2"]["beta"]
            pb["ffn"]["W1"] -= lr*gb["ffn"]["W1"]; pb["ffn"]["b1"] -= lr*gb["ffn"]["b1"]
            pb["ffn"]["W2"] -= lr*gb["ffn"]["W2"]; pb["ffn"]["b2"] -= lr*gb["ffn"]["b2"]

def train_model(kind, epochs, lr, seed=42):
    p = init_model(seed, kind)
    fwd = {"A":forward_A, "B":forward_B, "C":forward_C}[kind]
    bwd = {"A":backward_A, "B":backward_B, "C":backward_C}[kind]
    for epoch in range(1, epochs+1):
        total_loss = 0.0
        for ctx, tgt in train_pairs_sub:
            ids = [state_to_id[w] for w in ctx]; tid = state_to_id[tgt]
            probs, cache = fwd(ids, p)
            total_loss += -np.log(probs[tid]+1e-12)
            grads = bwd(ids, tid, probs, cache, p)
            apply_grads(p, grads, lr, kind)
        if epoch % 10 == 0 or epoch == 1:
            print(f"  [{kind}] Epoch {epoch:3d} | Avg Loss: {total_loss/len(train_pairs_sub):.4f}")
    return p

LR_BY_MODEL = {"A": 0.08, "B": 0.04, "C": 0.08}
results = {}
for kind in ["A","B","C"]:
    print(f"\n=== Training Model {kind} (lr={LR_BY_MODEL[kind]}) ===")
    results[kind] = train_model(kind, epochs=40, lr=LR_BY_MODEL[kind])



=== Training Model A (lr=0.08) ===
  [A] Epoch   1 | Avg Loss: 1.2366
  [A] Epoch  10 | Avg Loss: 0.0582
  [A] Epoch  20 | Avg Loss: 0.0517
  [A] Epoch  30 | Avg Loss: 0.0494
  [A] Epoch  40 | Avg Loss: 0.0482

=== Training Model B (lr=0.04) ===
  [B] Epoch   1 | Avg Loss: 2.4512
  [B] Epoch  10 | Avg Loss: 0.0045
  [B] Epoch  20 | Avg Loss: 0.0004
  [B] Epoch  30 | Avg Loss: 0.0002
  [B] Epoch  40 | Avg Loss: 0.0001

=== Training Model C (lr=0.08) ===
  [C] Epoch   1 | Avg Loss: 0.4502
  [C] Epoch  10 | Avg Loss: 0.0009
  [C] Epoch  20 | Avg Loss: 0.0004
  [C] Epoch  30 | Avg Loss: 0.0002
  [C] Epoch  40 | Avg Loss: 0.0002


## Part F- Results: Does Architecture Actually Matter?

We report both **training accuracy** (fits data seen during training) and **held-out
generalization accuracy** (the fair, no-collision test set from Part B) — plus log-loss,
which reveals confidence/calibration even when raw accuracy numbers look similar.

In [ ]:

def evaluate(kind, p, pairs):
    fwd = {"A":forward_A, "B":forward_B, "C":forward_C}[kind]
    correct, total_ll = 0, 0.0
    for ctx, tgt in pairs:
        ids = [state_to_id[w] for w in ctx]; tid = state_to_id[tgt]
        probs, _ = fwd(ids, p)
        correct += (np.argmax(probs) == tid)
        total_ll += -np.log(probs[tid]+1e-12)
    return correct/len(pairs), total_ll/len(pairs)

header = f"{'Model':6s} {'Train Acc':>10s} {'Train LL':>10s} {'Test Acc':>10s} {'Test LL':>10s}"
print(header)
print("-"*50)
for kind in ["A","B","C"]:
    tr_acc, tr_ll = evaluate(kind, results[kind], train_pairs_sub)
    te_acc, te_ll = evaluate(kind, results[kind], test_pairs_sub)
    print(f"{kind:6s} {tr_acc:>9.1%} {tr_ll:>10.4f} {te_acc:>9.1%} {te_ll:>10.4f}")


Model   Train Acc   Train LL   Test Acc    Test LL
--------------------------------------------------
A          97.1%     0.0422    100.0%     0.0606
B         100.0%     0.0001     75.3%     2.7441
C         100.0%     0.0002     94.0%     0.1505


### Reading this honestly

**Model B (bare attention) overfits.** 100% training accuracy, but generalization drops
to the mid-70s% with a very high log-loss (~2.7) — it memorized the training sequences'
surface patterns rather than the underlying transition rule. Without FFN/LayerNorm/residual
to regularize and stabilize it, it had nothing to prevent this.

**Model A (no attention at all) generalizes surprisingly well.** Because most of this
synthetic vocabulary's transitions are near-deterministic given just the last token, a
model with zero context mechanism still does very well — a genuinely humbling result:
architectural sophistication isn't automatically rewarded if the task doesn't need it.
This directly echoes the Week 4 mentor feedback: *"multi-head may have shown different
convergence behavior, but it did not improve final accuracy on this dataset."* Here,
the full Transformer block doesn't even guarantee beating the *simplest possible* model
on raw accuracy either.

**Model C (full Transformer block) is the most *reliable* of the three, not necessarily
the most accurate.** It matches or nearly matches Model A's generalization accuracy while
being dramatically better calibrated than Model B (much lower test log-loss) — it fits
training data perfectly without the instability or the overfitting collapse seen in Model
B. The honest claim is: **the added components bought stability and calibration, not a
guaranteed accuracy win on this particular dataset.**

## Part G- Spotlight: Watching the Models Handle a Genuinely Unseen Sequence

A concrete look at all three models facing the same never-seen-verbatim sequence,
starting from a word (`Restock`) that never appeared as a sequence's first token during
training.

In [ ]:

spotlight = [s for s in test_sequences if s[0] in ("Restock","Shipment") and "Forecast" in s][:3]
for seq in spotlight:
    fi = seq.index("Forecast")
    if fi+1 >= len(seq):
        continue
    ctx, true_next = seq[:fi+1], seq[fi+1]
    print(f"Full unseen sequence: {seq}")
    print(f"Context up to and including \'Forecast\': {ctx}  (true next word: {true_next})")
    for kind in ["A","B","C"]:
        ids = [state_to_id[w] for w in ctx]
        fwd = {"A":forward_A, "B":forward_B, "C":forward_C}[kind]
        probs, _ = fwd(ids, results[kind])
        pred = id_to_state[np.argmax(probs)]
        mark = "correct" if pred == true_next else "WRONG"
        print(f"  Model {kind}: predicted \'{pred}\'  ({mark})")
    print()


Full unseen sequence: ['Restock', 'Inventory', 'Forecast', 'Order', 'Shipment', 'Receive', 'Restock', 'Inventory', 'Forecast', 'Order']
Context up to and including 'Forecast': ['Restock', 'Inventory', 'Forecast']  (true next word: Order)
  Model A: predicted 'Order'  (correct)
  Model B: predicted 'Receive'  (WRONG)
  Model C: predicted 'Order'  (correct)

Full unseen sequence: ['Restock', 'Inventory', 'Forecast', 'Order', 'Shipment', 'Receive', 'Restock', 'Inventory', 'Forecast', 'Order', 'Shipment']
Context up to and including 'Forecast': ['Restock', 'Inventory', 'Forecast']  (true next word: Order)
  Model A: predicted 'Order'  (correct)
  Model B: predicted 'Receive'  (WRONG)
  Model C: predicted 'Order'  (correct)

Full unseen sequence: ['Shipment', 'Receive', 'Restock', 'Inventory', 'Forecast', 'Order', 'Shipment', 'Receive', 'Restock', 'Inventory']
Context up to and including 'Forecast': ['Shipment', 'Receive', 'Restock', 'Inventory', 'Forecast']  (true next word: Order)
  Model

Even Model C the most sophisticated of the three — gets one of these wrong in our
run. That's worth stating plainly rather than glossing over: a correct architecture and a
verified backward pass do not guarantee correct predictions on every unseen case. With
only 450 training pairs and 40 epochs of plain SGD, some genuinely unseen combinations
will still trip up any of these models.

## Part H - What Did Each New Component Actually Buy Us?

Per the assignment's own instruction to keep returning to this question — here's the
honest, evidence-based answer for each piece, not a generic textbook one:

**Feed-Forward Network.** Hard to isolate its effect alone in this experiment (we only
compared "no attention" vs. "bare attention" vs. "full block"), but conceptually: attention
mixes information *across* positions; the FFN then reshapes what a *single* position's
now-mixed representation actually means, independently per token. Without it, the model's
only nonlinear processing power would come from the attention softmax itself.

**LayerNorm.** This experiment gave us a direct, measured answer: **training stability**.
Model B (no LayerNorm) diverged to `NaN` at a learning rate Model C (with LayerNorm)
handled without issue. That's a concrete, reproducible finding, not a hypothesis.

**Residual connections.** Not isolated in this experiment either, but structurally: they
give every sub-layer an "easy" default (pass the input through unchanged) to fall back on
before learning something better — worth testing directly in a future week by training a
version of Model C with the residual connections removed and comparing convergence.

**Stacking 2 blocks (depth).** The assignment's hypothesis was that Block 1 might learn
simple associations (`Forecast` relates to `Inventory`) while Block 2 refines them further
(`Forecast` after `Inventory` alone means something different than `Forecast` after a full
`Restock → Inventory` build-up). We did not verify this claim directly — doing so honestly
would require inspecting each block's intermediate attention patterns separately, which
we have not done here. **Flagging this as an untested hypothesis, not a demonstrated
finding, per the Week 4 mentor feedback on separating observation from interpretation.**

**The 3-model comparison, overall.** The single clearest, most defensible finding from
this whole notebook: **the fancier architecture (Model C) did not straightforwardly "win"
on raw accuracy against the simplest one (Model A) on this dataset** — but it was
dramatically more stable to train and better calibrated than the mid-complexity option
(Model B). Architecture mattered for *how reliably* the model trained, not automatically
for *how accurate* the final result was. Whether that generalizes beyond this specific
synthetic dataset is exactly the kind of question that would need testing on a genuinely
different task before treating it as a general rule.